# **EECS 6895 - Assignment 3**

This notebook contains three parts:
- **Problem 1:** CRIT Socratic Reasoning
- **Problem 2:** Big Five Personality Classification
- **Problem 3:** Short conceptual questions

**Important Notes**
- Use a GPU runtime.
- Please leave all helper functions unchanged unless the notebook asks you to modify them.
- Only modify code in the TODO sections.
- Remember to fill in all short answer questions, including in Problems 1 and 2.
- For Problem 1, please ensure your outputs match the specified JSON schema exactly.

## **0. Setup**

**Please do not change any code in this section.**

In [1]:
# !pip -q install transformers accelerate datasets empath beautifulsoup4 lxml scikit-learn pandas numpy

In [2]:
import re
import json
import random
import textwrap
import xml.etree.ElementTree as ElementTree

import numpy as np
import pandas as pd
import requests
import torch

from datasets import load_dataset
from empath import Empath

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.multioutput import MultiOutputClassifier
from transformers import AutoModelForCausalLM, AutoTokenizer

In [3]:
RANDOM_SEED = 0
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
  torch.cuda.manual_seed_all(RANDOM_SEED)
else:
  raise Exception("Please switch to GPU runtime!")

In [4]:

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
  MODEL_NAME,
  trust_remote_code=True,
  torch_dtype=torch.float16,
  device_map="cuda",
)
model.eval()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((2048,), eps=1e-06)
    (ro

In [5]:
def generate_text(user_prompt, system_prompt="You are a careful assistant that exactly follows output formats.", max_new_tokens=256):
  messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
  ]
  text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
  inputs = tokenizer(text, return_tensors="pt").to(model.device)
  with torch.no_grad():
    outputs = model.generate(
      **inputs,
      max_new_tokens=max_new_tokens,
      do_sample=False,
      pad_token_id=tokenizer.eos_token_id,
    )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

def extract_first_json_block(text):
  text = text.strip()
  try:
    return json.loads(text)
  except Exception:
    pass
  fenced = re.search(r"```json\s*(\{.*?\}|\[.*?\])\s*```", text, flags=re.DOTALL)
  if fenced:
    return json.loads(fenced.group(1))
  for pattern in [r"(\{.*\})", r"(\[.*\])"]:
    m = re.search(pattern, text, flags=re.DOTALL)
    if m:
      return json.loads(m.group(1))
  raise ValueError("Could not parse JSON from model output. Raw output:\n" + text[:1000])

def pretty(obj):
  print(json.dumps(obj, indent=2, ensure_ascii=False))

## **1. Problem 1: CRIT Socratic Reasoning**
In this problem you will compare the following:
- **Single-shot** prompting baseline
- **Staged CRIT pipeline** with Definition, Elenchus, and Dialectic
- **Breadth-to-depth Socratic dialogue**
- **Low/high-contentiousness** rewrites

The source text is pulled directly from public  abstracts from:
- **SocraSynth: Multi-LLM Reasoning with Conditional Statistics**
- **EVINCE: Optimizing Multi-LLM Dialogues Using Conditional Statistics and Information Theory**

**Please fill out TODOs 1-9 in code and the 3 short responses about your findings.**

In [6]:
def fetch_arxiv_metadata(arxiv_id):
  url = f"https://export.arxiv.org/api/query?id_list={arxiv_id}"
  response = requests.get(url, timeout=30)
  response.raise_for_status()
  root = ElementTree.fromstring(response.text)
  ns = {"atom": "http://www.w3.org/2005/Atom"}
  entry = root.find("atom:entry", ns)
  if entry is None:
    raise ValueError(f"No arXiv entry found for {arxiv_id}")
  title = entry.find("atom:title", ns).text.strip().replace("\n", " ")
  summary = entry.find("atom:summary", ns).text.strip().replace("\n", " ")
  return {"arxiv_id": arxiv_id, "title": title, "text": summary}

passages = [
  fetch_arxiv_metadata("2402.06634"),
  fetch_arxiv_metadata("2408.14575"),
]

for passage in passages:
  print("=" * 80)
  print(passage["title"])
  print("arXiv:", passage["arxiv_id"])
  print(textwrap.fill(passage["text"], width=100))
  print()

SocraSynth: Multi-LLM Reasoning with Conditional Statistics
arXiv: 2402.06634
Large language models (LLMs), while promising, face criticisms for biases, hallucinations, and a
lack of reasoning capability. This paper introduces SocraSynth, a multi-LLM agent reasoning platform
developed to mitigate these issues. SocraSynth utilizes conditional statistics and systematic
context enhancement through continuous arguments, alongside adjustable debate contentiousness
levels. The platform typically involves a human moderator and two LLM agents representing opposing
viewpoints on a given subject. SocraSynth operates in two main phases: knowledge generation and
reasoning evaluation. In the knowledge generation phase, the moderator defines the debate topic and
contentiousness level, prompting the agents to formulate supporting arguments for their respective
stances. The reasoning evaluation phase then employs Socratic reasoning and formal logic principles
to appraise the quality of the arguments p

In [7]:
def first_n_sentences(text, n=2):
  parts = re.split(r"(?<=[.!?])\s+", text.strip())
  return " ".join(parts[:n])

socratic_topic = first_n_sentences(passages[1]["text"], n=2)
print(socratic_topic)

EVINCE (Entropy and Variation IN Conditional Exchanges) is a novel framework for optimizing multi-LLM dialogues using conditional statistics and information theory. It addresses limitations in multi-agent debate (MAS) frameworks, where multiple LLMs ``chat'' without behavior modulation or mutual information quality assessment.


### Required JSON schemas for Problem 1

**Please output your answers to the following subproblems in these formats.**

**Single-shot output**
```
{
  "main_claim": str,
  "supporting_reasons": [str, ...],   # length 2 or 3
  "evidence": [str, ...],             # same length as supporting_reasons "counterargument": str
}
```
  
**Definition output**
```
{
  "main_claim": str,
  "supporting_reasons": [str, ...]    # length 2 or 3
}
```

**Elenchus output**
```
{
  "reason": str,
  "evidence_text": str,
  "evidence_type": str,               # empirical, example, expert_opinion, conceptual_reasoning, none
  "validity_score": int,              # 1 to 5
  "credibility_score": int            # 1 to 5
}
```

**Dialectic output**
```
{
  "reason": str,
  "counterargument": str,
  "counter_strength": int             # 1 to 5
}
```

**Breadth output**
```
{
  "broad_questions": [str, str, str, str]
}
```

**Depth output**
```
{
  "selected_question": str,
  "deep_questions": [str, str, str],
  "focused_answer": str
}
```

**Contentiousness output**
```
{
  "contentiousness": str,             # "low" or "high"
  "response": str
}
```

In [8]:
# TODO 1: Write the single-shot prompt.
# The model must return a JSON object with the exact single-shot schema.

def build_single_shot_prompt(passage):
  return f"""Read the following passage and analyze it in a single step.

Passage: {passage['text']}

Return ONLY a JSON object with this exact structure (no extra text):
{{
  "main_claim": "<the main claim or conclusion of the passage>",
  "supporting_reasons": ["<reason 1>", "<reason 2>"],
  "evidence": ["<evidence for reason 1>", "<evidence for reason 2>"],
  "counterargument": "<one counterargument to the main claim>"
}}

Rules:
- supporting_reasons must have 2 or 3 items
- evidence must have the same number of items as supporting_reasons
- Output valid JSON only"""

In [9]:
# TODO 2: Write the Definition prompt.
# The model must return a JSON object with the exact Definition schema.

def build_definition_prompt(passage):
  return f"""Read the following passage and identify its main claim and supporting reasons.

Passage: {passage['text']}

Return ONLY a JSON object with this exact structure (no extra text):
{{
  "main_claim": "<the central thesis or conclusion of the passage>",
  "supporting_reasons": ["<reason 1>", "<reason 2>"]
}}

Rules:
- supporting_reasons must have 2 or 3 items
- Output valid JSON only"""

In [10]:
# TODO 3: Write the Elenchus prompt.
# The model must return a JSON object with the exact Elenchus schema.

def build_elenchus_prompt(passage, reason):
  return f"""You are evaluating a supporting reason extracted from a passage.

Passage: {passage['text']}

Reason to evaluate: {reason}

Examine the evidence behind this reason and assess its quality.

Return ONLY a JSON object with this exact structure (no extra text):
{{
  "reason": "{reason}",
  "evidence_text": "<quote or paraphrase of the evidence from the passage supporting this reason>",
  "evidence_type": "<one of: empirical, example, expert_opinion, conceptual_reasoning, none>",
  "validity_score": <integer 1 to 5>,
  "credibility_score": <integer 1 to 5>
}}

Rules:
- evidence_type must be exactly one of: empirical, example, expert_opinion, conceptual_reasoning, none
- validity_score and credibility_score must be integers between 1 and 5
- Output valid JSON only"""

In [11]:
# TODO 4: Write the Dialectic prompt.
# The model must return a JSON object with the exact Dialectic schema.

def build_dialectic_prompt(passage, reason):
  return f"""You are constructing a counterargument to a supporting reason from a passage.

Passage: {passage['text']}

Reason: {reason}

Generate a counterargument that challenges this reason.

Return ONLY a JSON object with this exact structure (no extra text):
{{
  "reason": "{reason}",
  "counterargument": "<a specific counterargument that challenges the reason above>",
  "counter_strength": <integer 1 to 5>
}}

Rules:
- counter_strength must be an integer between 1 (weak) and 5 (very strong)
- Use plain text only — do not use backslashes or special escape characters in any string value
- Output valid JSON only"""

In [12]:
def run_single_shot(passage):
  prompt = build_single_shot_prompt(passage)
  raw = generate_text(prompt, max_new_tokens=256)
  return extract_first_json_block(raw)

def run_definition(passage):
  prompt = build_definition_prompt(passage)
  raw = generate_text(prompt, max_new_tokens=192)
  return extract_first_json_block(raw)

def run_elenchus(passage, reason):
  prompt = build_elenchus_prompt(passage, reason)
  raw = generate_text(prompt, max_new_tokens=192)
  return extract_first_json_block(raw)

def run_dialectic(passage, reason):
  prompt = build_dialectic_prompt(passage, reason)
  raw = generate_text(prompt, max_new_tokens=192)
  return extract_first_json_block(raw)

In [13]:
# TODO 5: Implement the deterministic score aggregation function.
# Keep the final score on a 1 to 5 scale.

def compute_final_score(elenchus_outputs, dialectic_outputs):
  # Collect validity and credibility scores from elenchus
  validity_scores = [e.get("validity_score", 3) for e in elenchus_outputs]
  credibility_scores = [e.get("credibility_score", 3) for e in elenchus_outputs]
  # Collect counter_strength scores from dialectic (inverted: strong counter weakens the argument)
  counter_strengths = [d.get("counter_strength", 3) for d in dialectic_outputs]

  avg_validity = sum(validity_scores) / len(validity_scores) if validity_scores else 3
  avg_credibility = sum(credibility_scores) / len(credibility_scores) if credibility_scores else 3
  avg_counter = sum(counter_strengths) / len(counter_strengths) if counter_strengths else 3

  # Evidence quality: average of validity and credibility
  evidence_quality = (avg_validity + avg_credibility) / 2
  # Final score: evidence quality penalized by counter strength, clamped to [1, 5]
  raw_score = evidence_quality - 0.3 * (avg_counter - 3)
  return round(max(1.0, min(5.0, raw_score)), 2)

In [14]:
# TODO 6: Build a comparison table with one row per passage.
# Include at least: title, single_shot_claim, staged_claim, num_reasons, final_score.

def _safe_extract(raw):
  """Wrapper around extract_first_json_block that fixes invalid JSON escape sequences."""
  try:
    return extract_first_json_block(raw)
  except Exception:
    # Replace invalid \X escapes (anything not a valid JSON escape) with \\X
    cleaned = re.sub(r'\\(?!["\\/bfnrtu])', r'\\\\', raw)
    return extract_first_json_block(cleaned)

def _run_dialectic_safe(passage, reason):
  prompt = build_dialectic_prompt(passage, reason)
  raw = generate_text(prompt, max_new_tokens=192)
  return _safe_extract(raw)

def build_reasoning_comparison_table(passages):
  rows = []
  for passage in passages:
    # Single-shot pipeline
    single_shot = run_single_shot(passage)

    # Staged CRIT pipeline
    definition = run_definition(passage)
    reasons = definition.get("supporting_reasons", [])

    elenchus_outputs = [run_elenchus(passage, r) for r in reasons]
    dialectic_outputs = [_run_dialectic_safe(passage, r) for r in reasons]

    final_score = compute_final_score(elenchus_outputs, dialectic_outputs)

    rows.append({
      "title": passage["title"],
      "single_shot_claim": single_shot.get("main_claim", ""),
      "staged_claim": definition.get("main_claim", ""),
      "num_reasons": len(reasons),
      "final_score": final_score,
    })

  return pd.DataFrame(rows)

In [15]:
# TODO 7: Write the breadth prompt.
# The model must return a JSON object with the exact Breadth schema.

def build_breadth_prompt(topic_text):
  return f"""Given the following topic, generate four broad exploratory questions that cover different dimensions of the subject.

Topic: {topic_text}

Return ONLY a JSON object with this exact structure (no extra text):
{{
  "broad_questions": ["<question 1>", "<question 2>", "<question 3>", "<question 4>"]
}}

Rules:
- broad_questions must contain exactly 4 questions
- Each question should explore a different angle or dimension of the topic
- Output valid JSON only"""

In [16]:
# TODO 8: Write the depth prompt.
# The model must return a JSON object with the exact Depth schema.

def build_depth_prompt(topic_text, selected_question):
  return f"""You are conducting a focused Socratic inquiry. Starting from a selected broad question, generate deeper follow-up questions and provide a focused answer.

Topic: {topic_text}

Selected question: {selected_question}

Return ONLY a JSON object with this exact structure (no extra text):
{{
  "selected_question": "{selected_question}",
  "deep_questions": ["<deeper follow-up question 1>", "<deeper follow-up question 2>", "<deeper follow-up question 3>"],
  "focused_answer": "<a concise focused answer to the selected question based on the topic>"
}}

Rules:
- deep_questions must contain exactly 3 questions that drill deeper into the selected question
- focused_answer should be 1-3 sentences directly addressing the selected question
- Output valid JSON only"""

In [17]:
# TODO 9: Write the contentiousness-controlled prompt.
# The model must return a JSON object with the exact contentiousness schema.

def build_contentiousness_prompt(topic_text, contentiousness_level):
  if contentiousness_level == "low":
    style_instruction = (
      "Respond in a calm, collaborative, and neutral tone. "
      "Acknowledge multiple perspectives and use measured, diplomatic language. "
      "Avoid provocative or polarizing statements."
    )
  else:
    style_instruction = (
      "Respond in a bold, assertive, and challenging tone. "
      "Take a strong stance, use emphatic language, and highlight tensions and disagreements. "
      "Do not shy away from provocative claims."
    )

  return f"""Respond to the following topic with the specified contentiousness level.

Topic: {topic_text}

Contentiousness level: {contentiousness_level}
Style instruction: {style_instruction}

Return ONLY a JSON object with this exact structure (no extra text):
{{
  "contentiousness": "{contentiousness_level}",
  "response": "<your response to the topic in the specified tone>"
}}

Rules:
- contentiousness must be exactly "{contentiousness_level}"
- response should be 2-4 sentences
- Output valid JSON only"""

In [18]:
# Run the single-shot and staged pipelines on the two passages.
reasoning_df = build_reasoning_comparison_table(passages)
reasoning_df

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


,title,single_shot_claim,staged_claim,num_reasons,final_score
0,SocraSynth: Multi-LLM Reasoning with Condition...,SocraSynth is an effective multi-LLM agent rea...,"SocraSynth, a multi-LLM agent reasoning platfo...",2,4.85
1,EVINCE: Optimizing Multi-LLM Dialogues Using C...,EVINCE is a novel framework for optimizing mul...,EVINCE is a novel framework for optimizing mul...,3,5.00


**Short Response #1**  
In 1 to 3 sentences, compare the single-shot and staged CRIT pipelines. Which one appears more structured or simpler?

**Answer:** The staged CRIT pipeline is more structured: it separates claim extraction (Definition), evidence evaluation (Elenchus), and counterargument generation (Dialectic) into three distinct steps, each with a dedicated JSON schema. The single-shot approach compressed all of these into one prompt and produced similar main claims (e.g., both identified EVINCE as a framework for optimizing multi-LLM dialogue), but without the per-reason quality scores that CRIT provides; the staged pipeline additionally yielded quantified final scores of 4.85 and 5.00 for the two passages, making reasoning quality explicitly measurable.

In [19]:
breadth_prompt = build_breadth_prompt(socratic_topic)
breadth_output = extract_first_json_block(generate_text(breadth_prompt, max_new_tokens=192))
pretty(breadth_output)

{
  "broad_questions": [
    "How does EVINCE's use of entropy and variation in conditional exchanges improve the efficiency and effectiveness of multi-agent dialogue compared to existing MAS frameworks?",
    "What specific statistical methods and information theory principles underpin the operation of EVINCE, and how do they contribute to its ability to assess and enhance mutual information between LLMs in dialogue?",
    "In what ways can EVINCE be adapted or extended to handle more complex scenarios or larger numbers of LLM participants in multi-agent dialogues?",
    "How might the insights and methodologies developed through EVINCE influence future research on optimizing dialogue systems beyond just multi-agent debate?"
  ]
}


In [20]:
FOCUS_INDEX = 1
selected_question = breadth_output["broad_questions"][FOCUS_INDEX]

depth_prompt = build_depth_prompt(socratic_topic, selected_question)
depth_output = extract_first_json_block(generate_text(depth_prompt, max_new_tokens=256))
pretty(depth_output)

{
  "selected_question": "What specific statistical methods and information theory principles underpin the operation of EVINCE, and how do they contribute to its ability to assess and enhance mutual information between LLMs in dialogue?",
  "deep_questions": [
    "How does EVINCE utilize entropy to measure the unpredictability of responses in a dialogue?",
    "Can you explain how EVINCE incorporates conditional statistics to improve the quality of interactions between LLMs?",
    "What role does mutual information play in EVINCE's framework, and how does it help in assessing the effectiveness of LLMs in a dialogue?"
  ],
  "focused_answer": "EVINCE leverages statistical methods like entropy and mutual information, along with principles from information theory, to quantify and optimize the interactions between multiple LLMs in a dialogue, thereby enhancing their mutual understanding and effectiveness."
}


**Short response #2**  
In 1 to 3 sentences, explain how the dialogue moved from broad exploration to a more focused line of reasoning.

**Answer:** The dialogue began by generating four broad questions spanning distinct dimensions of EVINCE — its efficiency gains, its mathematical foundations, its scalability, and its influence on future research. Selecting the second question focused the inquiry onto a specific technical thread: the statistical and information-theoretic principles underlying EVINCE. The depth stage then drilled further into that thread with three targeted follow-up questions about entropy, conditional statistics, and mutual information, ultimately converging on a concise answer about how EVINCE uses these tools to quantify and optimize LLM interactions.

In [21]:
low_prompt = build_contentiousness_prompt(socratic_topic, "low")
high_prompt = build_contentiousness_prompt(socratic_topic, "high")

low_output = extract_first_json_block(generate_text(low_prompt, max_new_tokens=192))
high_output = extract_first_json_block(generate_text(high_prompt, max_new_tokens=192))

pretty(low_output)
print()
pretty(high_output)

{
  "contentiousness": "low",
  "response": "EVINCE offers a promising approach to enhance dialogue optimization among multiple large language models by incorporating conditional statistics and information theory. This framework aims to improve the interaction quality and mutual understanding between agents in multi-agent systems."
}

{
  "contentiousness": "high",
  "response": "EVINCE represents a groundbreaking approach to enhancing dialogue efficiency among multiple LLMs, yet its reliance on conditional statistics and information theory may overlook the nuanced human-like interactions essential for truly effective multi-agent debates. The framework's rigidity in assessing mutual information could lead to oversimplified exchanges, undermining the complexity and depth required for meaningful dialogue."
}


**Short response #3**  
In 1 to 3 sentences, compare the low-contentiousness and high-contentiousness responses. Focus on tone, emphasis, and language.

**Answer:** The low-contentiousness response used hedged, collaborative language ("offers a promising approach," "aims to improve"), presenting EVINCE neutrally without taking a strong stance. The high-contentiousness response used assertive framing ("groundbreaking approach") but immediately pivoted to sharp criticism, arguing that EVINCE's reliance on formal metrics may "overlook nuanced human-like interactions" and produce "oversimplified exchanges" — language that is emphatic and adversarial rather than descriptive.

## **2. Problem 2: Big Five Personality Classification from Text**

In this problem, you will build text-based models to predict the Big Five personality traits from essay data.

You will complete the following tasks:
1. Load and inspect the `essays-big5` dataset.
2. Build **TF-IDF** features from raw essay text.
3. Build **Empath** psycholinguistic category features.
4. Train three **multi-output classification** models:
   - TF-IDF only
   - Empath only
   - Combined TF-IDF and Empath
5. Perform an **ablation study** by removing one Empath category.
6. Evaluate all models using **Accuracy, F1-score, and ROC-AUC**.
7. Answer the short analysis questions at the end.

**Please fill out TODOs 10-15 in code and the 2 short responses about your findings.**

In [22]:
# Load the public essays-big5 dataset.
dataset = load_dataset("jingjietan/essays-big5")
print(dataset)

# Build a single DataFrame, then create train/val/test partitions.
frames = []
for split_name in dataset.keys():
  df_split = dataset[split_name].to_pandas()
  df_split["source_split"] = split_name
  frames.append(df_split)

df = pd.concat(frames, ignore_index=True)
print(df.head(2))
print(df.columns.tolist())

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/795k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/953k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1578 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/395 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/494 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['O', 'C', 'E', 'A', 'N', 'ptype', 'text', '__index_level_0__'],
        num_rows: 1578
    })
    validation: Dataset({
        features: ['O', 'C', 'E', 'A', 'N', 'ptype', 'text', '__index_level_0__'],
        num_rows: 395
    })
    test: Dataset({
        features: ['O', 'C', 'E', 'A', 'N', 'ptype', 'text', '__index_level_0__'],
        num_rows: 494
    })
})
   O  C  E  A  N ptype                                               text  \
0  1  0  0  1  1    19  it is wednesday. I can't wait until friday bec...   
1  1  1  1  0  1    29  wow, I want to go talk to the socialist organi...   

   __index_level_0__ source_split  
0                774        train  
1                178        train  
['O', 'C', 'E', 'A', 'N', 'ptype', 'text', '__index_level_0__', 'source_split']


In [23]:
# Standardize column names if needed.
text_col_candidates = [c for c in df.columns if c.lower() in ["text", "essay", "essays"]]
if not text_col_candidates:
  raise ValueError(f"Could not find essay text column. Columns: {df.columns.tolist()}")
TEXT_COL = text_col_candidates[0]

trait_map = {}
for trait in ["O", "C", "E", "A", "N"]:
  matches = [c for c in df.columns if c.lower() == trait.lower()]
  if not matches:
    raise ValueError(f"Could not find trait column {trait}. Columns: {df.columns.tolist()}")
  trait_map[trait] = matches[0]

# Keep required columns and remove missing text.
work_df = df[[TEXT_COL] + [trait_map[t] for t in ["O", "C", "E", "A", "N"]]].copy()
work_df = work_df.rename(columns={TEXT_COL: "text", **trait_map})
work_df = work_df.dropna(subset=["text"]).reset_index(drop=True)

# Binarize trait labels.
for trait in ["O", "C", "E", "A", "N"]:
  work_df[trait] = work_df[trait].astype(int)

# Deterministic split: 70/15/15.
perm = np.random.RandomState(RANDOM_SEED).permutation(len(work_df))
work_df = work_df.iloc[perm].reset_index(drop=True)

n = len(work_df)
n_train = int(0.70 * n)
n_val = int(0.15 * n)

train_df = work_df.iloc[:n_train].reset_index(drop=True)
val_df = work_df.iloc[n_train:n_train + n_val].reset_index(drop=True)
test_df = work_df.iloc[n_train + n_val:].reset_index(drop=True)

print(f"train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

train=1726, val=370, test=371


In [24]:
# TODO 10: Display dataset statistics and label balance information. Use the columns "split", "num_examples", "avg_chars", "{O/C/E/A/N}_positive_rate".

def summarize_split(name, split_df):
  row = {
    "split": name,
    "num_examples": len(split_df),
    "avg_chars": round(split_df["text"].str.len().mean(), 1),
  }
  for trait in ["O", "C", "E", "A", "N"]:
    row[f"{trait}_positive_rate"] = round(split_df[trait].mean(), 4)
  return row

stats_df = pd.DataFrame([
  summarize_split("train", train_df),
  summarize_split("val", val_df),
  summarize_split("test", test_df),
])

stats_df

,split,num_examples,avg_chars,O_positive_rate,C_positive_rate,E_positive_rate,A_positive_rate,N_positive_rate
0,train,1726,3287.5,0.5180,0.4919,0.5261,0.5307,0.5139
1,val,370,3342.7,0.5514,0.5703,0.4973,0.5459,0.4405
2,test,371,3290.9,0.4663,0.5202,0.4960,0.5175,0.4933


In [25]:
# TODO 11: Build TF-IDF features from raw essay text. Do not use precomputed features.

def build_tfidf_features(train_texts, val_texts, test_texts):
  vectorizer = TfidfVectorizer(
    max_features=10000,
    sublinear_tf=True,
    min_df=2,
    ngram_range=(1, 2),
  )
  X_train = vectorizer.fit_transform(train_texts)
  X_val = vectorizer.transform(val_texts)
  X_test = vectorizer.transform(test_texts)
  return vectorizer, X_train, X_val, X_test

vectorizer, X_train_tfidf, X_val_tfidf, X_test_tfidf = build_tfidf_features(
  train_df["text"], val_df["text"], test_df["text"]
)

print(X_train_tfidf.shape, X_val_tfidf.shape, X_test_tfidf.shape)

(1726, 10000) (370, 10000) (371, 10000)


In [26]:
lexicon = Empath()
EMPATH_CATEGORIES = [
  "achievement",
  "social_media",
  "friends",
  "work",
  "positive_emotion",
  "negative_emotion",
  "help",
  "communication",
  "reading",
  "thinking",
]

In [27]:
# TODO 12: Build Empath-based category features. Use lexicon.analyze(text, categories=EMPATH_CATEGORIES, normalize=True).

def build_empath_features(texts):
  rows = []
  for text in texts:
    scores = lexicon.analyze(text, categories=EMPATH_CATEGORIES, normalize=True)
    # If analyze returns None (empty text), fill with zeros
    if scores is None:
      scores = {cat: 0.0 for cat in EMPATH_CATEGORIES}
    rows.append([scores.get(cat, 0.0) for cat in EMPATH_CATEGORIES])
  return np.array(rows, dtype=np.float32)

X_train_empath = build_empath_features(train_df["text"])
X_val_empath = build_empath_features(val_df["text"])
X_test_empath = build_empath_features(test_df["text"])

# Provided code: scale Empath feature dimensions using train statistics.
empath_scaler = StandardScaler()
X_train_empath = empath_scaler.fit_transform(X_train_empath)
X_val_empath = empath_scaler.transform(X_val_empath)
X_test_empath = empath_scaler.transform(X_test_empath)

print(X_train_empath.shape, X_val_empath.shape, X_test_empath.shape)

(1726, 10) (370, 10) (371, 10)


In [28]:
y_train = train_df[["O", "C", "E", "A", "N"]].to_numpy(dtype=np.int64)
y_val = val_df[["O", "C", "E", "A", "N"]].to_numpy(dtype=np.int64)
y_test = test_df[["O", "C", "E", "A", "N"]].to_numpy(dtype=np.int64)

X_train_combined = hstack([X_train_tfidf, csr_matrix(X_train_empath)])
X_val_combined = hstack([X_val_tfidf, csr_matrix(X_val_empath)])
X_test_combined = hstack([X_test_tfidf, csr_matrix(X_test_empath)])

In [29]:
# TODO 13: Train a multi-output logistic regression classifier.

def fit_multioutput_logistic(X_train, y_train):
  clf = MultiOutputClassifier(
    LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs", random_state=RANDOM_SEED),
    n_jobs=-1,
  )
  clf.fit(X_train, y_train)
  return clf

tfidf_model = fit_multioutput_logistic(X_train_tfidf, y_train)
empath_model = fit_multioutput_logistic(X_train_empath, y_train)
combined_model = fit_multioutput_logistic(X_train_combined, y_train)

In [30]:
# TODO 14: Evaluate all models using Accuracy, F1-score, and ROC-AUC. Report per-trait values and the mean across traits.

TRAITS = ["O", "C", "E", "A", "N"]

def safe_auc(y_true, y_score):
  if len(np.unique(y_true)) < 2:
    return 0.5
  return roc_auc_score(y_true, y_score)

def evaluate_model(model, X_test, y_test, model_name):
  y_pred = model.predict(X_test)
  y_prob = np.column_stack([
    est.predict_proba(X_test)[:, 1] for est in model.estimators_
  ])

  rows = []
  for i, trait in enumerate(TRAITS):
    acc = accuracy_score(y_test[:, i], y_pred[:, i])
    f1 = f1_score(y_test[:, i], y_pred[:, i], zero_division=0)
    auc = safe_auc(y_test[:, i], y_prob[:, i])
    rows.append({
      "model": model_name,
      "trait": trait,
      "accuracy": round(acc, 4),
      "f1": round(f1, 4),
      "roc_auc": round(auc, 4),
    })

  # Mean row across all traits
  rows.append({
    "model": model_name,
    "trait": "mean",
    "accuracy": round(np.mean([r["accuracy"] for r in rows]), 4),
    "f1": round(np.mean([r["f1"] for r in rows]), 4),
    "roc_auc": round(np.mean([r["roc_auc"] for r in rows]), 4),
  })

  return pd.DataFrame(rows)

results_df = pd.concat([
  evaluate_model(tfidf_model, X_test_tfidf, y_test, "TF-IDF"),
  evaluate_model(empath_model, X_test_empath, y_test, "Empath"),
  evaluate_model(combined_model, X_test_combined, y_test, "Combined"),
], ignore_index=True)

results_df

,model,trait,accuracy,f1,roc_auc
0,TF-IDF,O,0.6146,0.6166,0.6745
1,TF-IDF,C,0.5687,0.5580,0.5988
2,TF-IDF,E,0.5553,0.5985,0.5971
3,TF-IDF,A,0.5391,0.6275,0.5613
4,TF-IDF,N,0.6146,0.6187,0.6418
5,TF-IDF,mean,0.5785,0.6039,0.6147
6,Empath,O,0.5229,0.5542,0.5572
7,Empath,C,0.5633,0.5500,0.5691
8,Empath,E,0.5067,0.5850,0.5327
9,Empath,A,0.5472,0.6585,0.5812


In [31]:
# TODO 15: Perform an ablation study by removing one Empath category feature.

ablation_rows = []
for drop_idx, drop_cat in enumerate(EMPATH_CATEGORIES):
  # Build Empath features without the dropped category
  keep_indices = [i for i, cat in enumerate(EMPATH_CATEGORIES) if cat != drop_cat]

  X_train_abl = X_train_empath[:, keep_indices]
  X_test_abl = X_test_empath[:, keep_indices]

  abl_model = fit_multioutput_logistic(X_train_abl, y_train)

  y_pred = abl_model.predict(X_test_abl)
  y_prob = np.column_stack([
    est.predict_proba(X_test_abl)[:, 1] for est in abl_model.estimators_
  ])

  mean_acc = np.mean([accuracy_score(y_test[:, i], y_pred[:, i]) for i in range(5)])
  mean_f1 = np.mean([f1_score(y_test[:, i], y_pred[:, i], zero_division=0) for i in range(5)])
  mean_auc = np.mean([safe_auc(y_test[:, i], y_prob[:, i]) for i in range(5)])

  ablation_rows.append({
    "model": f"Empath_drop_{drop_cat}",
    "trait": "mean",
    "accuracy": round(mean_acc, 4),
    "f1": round(mean_f1, 4),
    "roc_auc": round(mean_auc, 4),
  })

empath_ablation_results = pd.DataFrame(ablation_rows)
empath_ablation_results

,model,trait,accuracy,f1,roc_auc
0,Empath_drop_achievement,mean,0.5348,0.5872,0.5649
1,Empath_drop_social_media,mean,0.5299,0.5771,0.5589
2,Empath_drop_friends,mean,0.5332,0.5901,0.5544
3,Empath_drop_work,mean,0.5305,0.5814,0.5602
4,Empath_drop_positive_emotion,mean,0.5321,0.5819,0.5575
5,Empath_drop_negative_emotion,mean,0.5272,0.5800,0.5465
6,Empath_drop_help,mean,0.5326,0.5842,0.5624
7,Empath_drop_communication,mean,0.5224,0.5746,0.5552
8,Empath_drop_reading,mean,0.5364,0.5891,0.5645
9,Empath_drop_thinking,mean,0.5337,0.5840,0.5620


In [32]:
comparison_df = pd.concat([results_df, empath_ablation_results], ignore_index=True)
comparison_df

,model,trait,accuracy,f1,roc_auc
0,TF-IDF,O,0.6146,0.6166,0.6745
1,TF-IDF,C,0.5687,0.5580,0.5988
2,TF-IDF,E,0.5553,0.5985,0.5971
3,TF-IDF,A,0.5391,0.6275,0.5613
4,TF-IDF,N,0.6146,0.6187,0.6418
5,TF-IDF,mean,0.5785,0.6039,0.6147
6,Empath,O,0.5229,0.5542,0.5572
7,Empath,C,0.5633,0.5500,0.5691
8,Empath,E,0.5067,0.5850,0.5327
9,Empath,A,0.5472,0.6585,0.5812


**Short Response #1**

Compare the three models. Which performs best, and what does this suggest about the relevance of TF-IDF versus psycholinguistic features?

**Answer:** TF-IDF performs best overall (mean accuracy 0.578, ROC-AUC 0.615), outperforming both Empath (0.534 / 0.562) and the Combined model (0.573 / 0.608). Notably, adding Empath features to TF-IDF did not improve performance — the Combined model is slightly weaker than TF-IDF alone — suggesting that the 10 psycholinguistic category scores add noise relative to the 10,000-dimensional lexical representation, and that surface-level word patterns carry more discriminative signal than category-level semantic aggregation for this dataset.

**Short Response #2**

What does the ablation study suggest about feature importance?

**Answer:** Removing `communication` caused the largest drop in accuracy (0.5224 vs baseline 0.5337), and removing `negative_emotion` caused the largest drop in ROC-AUC (0.5465 vs baseline 0.5620), identifying these two categories as the most informative features for Big Five prediction. Conversely, dropping `reading` or `achievement` slightly improved accuracy, suggesting these categories are redundant or mildly noisy for this task; overall, the ablation confirms that no single Empath category dominates, and the model's modest performance reflects the limited coverage of only 10 psycholinguistic dimensions.

## **3. Problem 3: Conceptual Questions**

Answer each question in 1 to 3 sentences.

### Q1. Socratic Reasoning
(a) Why can dialogic and multi-agent reasoning work better than monologue-style responses?  

**Answer:** Dialogic and multi-agent reasoning forces each agent to defend its position against challenges from others, which exposes hidden assumptions and logical gaps that a single monologue never surfaces. By assigning agents to opposing viewpoints and requiring iterative exchange, the system aggregates diverse perspectives and applies external pressure that drives toward more rigorous, well-examined conclusions.

(b) What are the roles of **definition**, **elenchus**, and **dialectic** in structured reasoning?

**Answer:** Definition establishes the shared foundation by precisely identifying the main claim and its supporting reasons, ensuring all parties are arguing about the same thing. Elenchus cross-examines each reason by scrutinizing the quality and type of evidence behind it, assigning validity and credibility scores to surface weaknesses. Dialectic then stress-tests the argument by generating targeted counterarguments and rating their strength, completing the adversarial loop that separates well-supported claims from fragile ones.

### Q2. Contentiousness
(a) How does contentiousness affect **tone**, **emphasis**, and **language** in generated responses?  

**Answer:** High contentiousness shifts tone from collaborative to adversarial, with emphasis placed on contradictions, limitations, and strong claims rather than on shared ground. Language becomes more assertive and polarizing — hedging phrases like "may offer" are replaced by emphatic statements and direct challenges — while low contentiousness favors measured, diplomatic phrasing that acknowledges multiple valid perspectives without committing to a strong stance.

(b) How might low-contentiousness and high-contentiousness reasoning be useful in different settings?

**Answer:** Low-contentiousness reasoning is appropriate for consensus-building, policy mediation, or collaborative problem-solving where the goal is to find common ground and avoid alienating stakeholders. High-contentiousness reasoning is valuable in adversarial stress-testing, debate training, or critical review settings where exposing the weakest points of an argument quickly is more important than maintaining a cooperative tone.

### Q3. Emotion and Representation
(a) Why would we model emotion as a **spectrum** rather than as a single discrete variable?

**Answer:** Emotions in natural language rarely map cleanly onto a single discrete label — a sentence can simultaneously express mild joy and underlying anxiety, or blend surprise with disgust. Modeling emotion as a spectrum over continuous dimensions (e.g., valence and arousal) or as a distribution over multiple categories captures this co-occurrence and intensity variation, yielding a richer representation that better reflects the graded, overlapping nature of real human affect.

(b) How is this idea related to **Plutchik's Wheel of Emotions** or **BEAM**?

**Answer:** Plutchik's Wheel arranges eight primary emotions in a circular structure where adjacent emotions blend into secondary ones and distance encodes opposition, making emotion inherently a matter of degree and mixture rather than a discrete choice. BEAM extends this by mapping linguistic expressions onto a probabilistic distribution across emotional dimensions, allowing a single utterance to be represented as a weighted combination of emotional states — precisely the spectrum view that a single discrete label cannot capture.

### Q4. Traits and Text
(a) Why are psycholinguistic or category-based features useful for predicting personality from text?  

**Answer:** Psycholinguistic features map raw vocabulary onto semantically meaningful categories — such as achievement, social behavior, or negative emotion — that align with the psychological constructs underlying personality traits, making the model's input more interpretable and theoretically grounded than raw word counts. They also aggregate sparse lexical signals into dense, stable scores that generalize better across writing styles and topics than individual high-frequency words.

(b) What are the limitations of inferring human traits from only language?

**Answer:** Language is a filtered, context-dependent expression of personality — people write differently depending on audience, platform, and topic, so a single essay may not reflect stable underlying traits. Text also cannot capture non-verbal signals (facial expression, prosody, behavior) that are central to personality assessment, and self-report bias means authors may consciously or unconsciously present idealized versions of themselves rather than authentic trait expression.

### Q5. Multimodal Perception
(a) Why might a **multimodal** sentiment system outperform a text-only or visual-only system?

**Answer: TODO**

(b) In class, we saw how to represents visual sentiment concepts using **adjective-noun pairs**. Why is this a better design than using nouns alone?

**Answer: TODO**